# OpenKind — T4 Classifier Environment Recovery
**Version 4.1.1 · 1 October 2026**  
`openkind-t4-recovery-reliability/v4.1.1`

This repairs the `ensurepip` failure in the v4.1.0 run. It **does not edit or resume that run**.
Default scope is **17 classifier stages**, not a repeat of the successful/continuing native and Go work.

**Let the old campaign finish and finalize first. Do not execute both campaigns in one live kernel.**
Then select a T4, leave the controls below, and choose **Runtime → Run all**.
The original 40-stage program is still available through the explicit optional track controls.

The repair creates environments without ensurepip, seeds a hash-pinned pip into the **explicit
worker interpreter**, and verifies local pip, Python ABI/prefix, inherited CUDA packages and imports.
Incomplete old environments and native/model caches are left intact.

**No new pretrained-model measurements are bundled.** CPU checks are separate from full worker
installation, GPU API checks and actual classifier training/evaluation.

In [ ]:
# 1. Controls — default: classifiers only. Optional tracks must be chosen before run creation.
CPU_VALIDATE_ONLY = False
SESSION_MINUTES = 360             # Work ceiling, not a completion estimate.
BUILD_TIMEOUT_MINUTES = 120
BUILD_JOBS = 0                   # Automatic conservative compiler parallelism.
RUN_CLASSIFIERS = True            # DeBERTa, ModernBERT frozen/fit, BERT frozen/fit.
RUN_GLICLASS = True               # Its own environment and capability gates.
RUN_INDECIS_REUSE = False          # Requires the original saved open-model weights.
RUN_NATIVE = False
RUN_NEW_HISTORY = False           # NF/PF: mixed, native-only, separate processes.
INCLUDE_4B = False                # Additional NJ/NF/PF + NLI; 9B is the default.
RUN_PARENT_POLICY_REPLAY = False   # Explicit post-hoc CPU arithmetic, no new timing.
RESUME_RUN = ""                  # Only an incomplete *v4.1.1* run may go here.

LAB = "/content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab"
PARENT_RUN = LAB + "/integrated_research_runs/20261001T105345_828556Z_6e6d92"
LOCAL_CACHE = "/content/openkind9b_t4l4_cache"
if RESUME_RUN and "recovery_reliability_runs/" not in RESUME_RUN:
    raise ValueError("Resume must refer to an incomplete recovery_reliability_runs run, not the v4.0.0 parent.")

## Repair boundary
Both saved failures occurred inside `venv`'s automatic `ensurepip` child. The logs do not reveal
its inner cause. This version uses `--without-pip`, then pip's documented `--python` targeting,
with pip 26.2.1's published wheel SHA-256. It never treats a half-created interpreter or global pip
visibility as a healthy worker. The coordinator is not upgraded by this setup.

Classifier dependency versions, fitting data/recipe and all model readouts/gates remain unchanged.
The prior same-file log repair remains in place. New `r411_*` environment identities do not overwrite
old `r41_*` directories. [Python venv](https://docs.python.org/3.13/library/venv.html) ·
[pip target interpreter](https://pip.pypa.io/en/stable/topics/python-option/).

This is a new code and run identity. `RESUME_RUN` must initially be empty.

In [ ]:
# 2. Verify and unpack the embedded source into a content-addressed directory.
from pathlib import Path
import base64, hashlib, io, json, os, sys, tempfile, zipfile, importlib
SOURCE_ZIP_SHA256 = "28c5b5a61a5939549d8b0d120853eeb076570598cf4e624e45c51c20b98e1ac5"
SOURCE_ZIP_B64 = "..."
raw = base64.b64decode(SOURCE_ZIP_B64)
if hashlib.sha256(raw).hexdigest() != SOURCE_ZIP_SHA256:
    raise RuntimeError("Embedded source archive hash mismatch")
WORK = Path(tempfile.mkdtemp(prefix="openkind_r411_cpu_")) if CPU_VALIDATE_ONLY else Path(LOCAL_CACHE)
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK / ("runner_r411_" + SOURCE_ZIP_SHA256[:16])
SOURCE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("Embedded source ZIP failed its integrity check")
    for member in archive.infolist():
        relative = Path(member.filename)
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Unsafe source archive path")
        target = SOURCE / relative
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        content = archive.read(member.filename)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError("Content-addressed source was edited: " + str(target))
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(content)
manifest = json.loads((SOURCE / "SOURCE_MANIFEST.json").read_text())
for name, expected in manifest.items():
    if hashlib.sha256((SOURCE / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Source manifest mismatch: " + name)
# Reload only this project's module names, avoiding stale code after a notebook update.
for path in SOURCE.glob("*.py"):
    sys.modules.pop(path.stem, None)
sys.path.insert(0, str(SOURCE))
importlib.invalidate_caches()
print("Verified source:", SOURCE)
print("Source archive SHA-256:", SOURCE_ZIP_SHA256)

In [ ]:
# 3. Coordinator dependencies and real CPU regression tests.
import importlib.util, subprocess, sys, os
required = {
    "numpy": "numpy>=2.0,<3", "pandas": "pandas>=2.2,<3",
    "sklearn": "scikit-learn>=1.6,<2", "tabulate": "tabulate>=0.9,<1",
    "packaging": "packaging>=24", "matplotlib": "matplotlib>=3.9,<4",
}
if not CPU_VALIDATE_ONLY:
    required.update({"huggingface_hub":"huggingface_hub>=1.5,<2", "tokenizers":"tokenizers>=0.23.1,<0.24"})
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    if CPU_VALIDATE_ONLY:
        raise RuntimeError("CPU validation dependencies missing: " + repr(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *missing], check=True)
# HF_TOKEN is optional; never print it or write it to evidence.
if not CPU_VALIDATE_ONLY and not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
        del token
    except Exception:
        print("HF_TOKEN unavailable; public model access will use unauthenticated requests.")
TEST_LOG = WORK / ("cpu_validation_r411_" + SOURCE_ZIP_SHA256[:16] + ".log")
with TEST_LOG.open("w") as stream:
    result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-p", "test_*.py"],
                            cwd=SOURCE, stdout=stream, stderr=subprocess.STDOUT, timeout=600)
print("\n".join(TEST_LOG.read_text(errors="replace").splitlines()[-12:]))
if result.returncode:
    raise RuntimeError("CPU validation failed; inspect " + str(TEST_LOG))
print("CPU checks passed. No pretrained inference was performed by these checks.")

## Default recovered experiments

| Classifier | Retained comparison |
|---|---|
| DeBERTa | Released three-way NLI |
| ModernBERT-base | Trained heads over a frozen backbone; separate full fine-tuning |
| BERT-base | Same two fitting modes as a historical control |
| GLiClass | Released dynamic-label model in its own worker environment |

These are different task/training contracts. Fitting still uses only the original eligible train
and development records. Old U4 confirmations remain recovery/regression data; the unchanged
`r41_*` targeted splits are retained. No labels or model settings are modified to fix installation.

The optional original native NF/PF and indecis tracks remain in source but are off by default.
They must not be reported as measured by a classifier-only run. See the embedded contract for
complete model, calibration, history and evidence limits.

In [ ]:
# 4. Freeze the requested program and inspect its default stage preview.
import copy
import r41_runner as runner
import openkind9b_core as core
import pandas as pd
from IPython.display import display
CONFIG = copy.deepcopy(runner.DEFAULTS)
CONFIG.update(run_encoders=RUN_CLASSIFIERS, run_gliclass=RUN_GLICLASS,
              run_indecis_reuse=RUN_INDECIS_REUSE, run_native=RUN_NATIVE,
              run_history=RUN_NEW_HISTORY, models=["9B", "4B"] if INCLUDE_4B else ["9B"],
              run_parent_policy_replay=RUN_PARENT_POLICY_REPLAY)
print("Protocol:", runner.PROTOCOL)
preview = json.loads((SOURCE / "assets" / "R411_CLASSIFIER_PLAN_PREVIEW.json").read_text())
print("Default classifier-only stages:", len(preview), "(actual plan is created from your controls and source data below)")
display(pd.DataFrame(preview)[["id", "kind", "requested_cases"]])

## Drive, resource and identity rules
The U4 `PARENT_RUN` remains an immutable source. Its input/source checks are unchanged.
The failed v4.1.0 run is not a resume destination. All new results are saved under a new identity.

The default run does not compile or load native Qwen and does not repeat indecis. Leave those tracks
off while recovering the classifiers. Enabling them requests the corresponding additional stages;
it does not reuse previously completed results under a new name.

Workers load one GPU classifier at a time and retain the no-silent-CPU-fallback checks.
Do not start this notebook while the prior notebook still has work running in the same runtime.

In [ ]:
# 5. Mount Drive and create a NEW identity (or validate an incomplete v4.1.1 identity).
STUDY = None
if CPU_VALIDATE_ONLY:
    print("SKIPPED: Drive mount, actual GPU admission, model work and evidence-run creation.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(PARENT_RUN).is_dir():
        raise FileNotFoundError("Parent evidence directory missing: " + PARENT_RUN)
    snapshot = core.gpu_snapshot()
    print("GPU:", snapshot["name"], "total GiB:", snapshot["total_gib"])
    if snapshot["total_gib"] < 14:
        raise RuntimeError("This T4-oriented GPU program requires at least 14 GiB reported total VRAM.")
    STUDY = runner.create(LAB, WORK, CONFIG, PARENT_RUN, minutes=SESSION_MINUTES,
                          resume=RESUME_RUN, cpu_validate=False,
                          build_minutes=BUILD_TIMEOUT_MINUTES, build_jobs=BUILD_JOBS)
    core.copy_artifact(TEST_LOG, STUDY.folder / "CPU_VALIDATION.log")
    STUDY.write("notebook_bundle_identity.json", {"version":"4.1.1", "source_zip_sha256":SOURCE_ZIP_SHA256})
    requested = runner.plan(CONFIG, STUDY.data)
    print("Requested stages:", len(requested), "Dataset records:", len(STUDY.data))
    print("New evidence:", STUDY.folder)
    display(pd.DataFrame([{k:v for k,v in s.items() if k != "case_ids"} for s in requested])[["id", "kind"]])

In [ ]:
# 6. Run. Each failed backend leaves independent stages eligible; budgets preserve completed blocks.
OUTCOME = None
if CPU_VALIDATE_ONLY:
    print("SKIPPED: package installation into model workers, pretrained downloads/loading/training, Go build, CUDA build, and real benchmarks.")
else:
    OUTCOME = runner.run_all(STUDY)
    print("Execution status:", OUTCOME["status"])
    print("Semantic and history conclusions require reading the separate tables/gates.")

## Read the outcome without overclaiming it
`execution_gate` is not an accuracy target. Quality stages use sentinel execution checks; history stages
use available distributions, selected answers and fixed diagnostic thresholds. The newly fitted acceptance
policy has **not** received a production history/concurrency/soak qualification just because those checks pass.

Read **REPORT.md**, **stage_status.json**, **quality_summary.csv**, **targeted_strata.csv**,
**paired_comparisons.json**, **request_policies.json**, **isolation_summary.csv**, and **indecis_reuse_summary.csv**.
`parent_NJ_policy_replay.json` is explicitly post-hoc CPU arithmetic, with no new inference/timing.
`historical_reference_quality.csv` is retained separately and is never pooled into current timings.

The wider question register remains open. This program does not implement a mixed-process root-cause repair,
a hard question-isolation mask, document-retention recovery, Rust/MLX qualification, cache tenant/TTL proof,
independent human evaluation, or release promotion.

In [ ]:
# 7. Compact handoff — execution status and scientific evidence remain separate.
if CPU_VALIDATE_ONLY:
    print("CPU validation complete. No new GPU scientific result or promotion is asserted.")
elif OUTCOME is not None:
    table = pd.DataFrame(OUTCOME.get("quality", []))
    if len(table):
        confirmation = table[table["panel"].str.contains("confirmation", regex=False)]
        columns = [k for k in ["profile","panel","unique_cases","field_accuracy","all_correct",
                               "request_p50_ms","probability_coverage","input_coverage","evidence_role"] if k in table]
        display(confirmation[columns])
    stages = core.load_json(STUDY.folder / "stage_status.json")
    display(pd.DataFrame(stages)[["id","status","execution_gate","execution_gate_scope"]])
    print("Protocol:", runner.PROTOCOL)
    print("Run:", STUDY.folder.name, "Status:", OUTCOME["status"])
    print("Evidence:", STUDY.folder)
    print("Archive:", STUDY.folder.with_suffix(".zip"))
    print("Completion marker:", (STUDY.folder/"COMPLETE.json").is_file())
    if OUTCOME["status"] == "EXPLORATORY_COMPLETE":
        print("Do not reuse this completed run ID. Read scientific gates separately.")
    else:
        print("Resume only with unchanged v4.1.1 code/config/hardware/backend identity:")
        print("RESUME_RUN =", repr(str(STUDY.folder)))